# GazeDecoder V3 — Multimodal Context Ablation (CtxV2)

**目标：用最小改动快速判断新增的多模态上下文（OCR / 图像 patch / 图像 page）是否对性能有帮助。**

## 实验设计

| 实验组 | ctx_mode | 上下文内容 | 维度（投影前） |
|--------|----------|-----------|--------------|
| `CtxV1`（基线） | `v1_compat` | embed_func(384) + embed_code(768) | 1152d |
| `CtxV2_Full`（全模态） | `full` | G1+G2+G3a+G3b+G4+G5 | 2688d |
| `CtxV2_NoImg`（纯文字） | `no_img` | G1+G2+G3a+G3b（去掉图像） | 1920d |
| `CtxV2_ImgOnly`（纯图像） | `img_only` | G4 patch + G5 page | 768d |

所有组共享：
- 模型架构：`Bchan_Spatial_CtxSA`（消融最优变体，F1_issue ≈ 0.9467）
- 训练协议：LOSO × 40 epochs，best-val-F1 checkpoint，seed=42
- 投影方式：固定随机正交矩阵将各模态 concat 向量压缩到 768d，**模型权重无需修改**

**判断标准：**
- `CtxV2_Full` vs `CtxV1`：总体上有无提升（决定是否值得做扩展消融）
- `CtxV2_NoImg` vs `CtxV1`：OCR 文本单独有没有价值
- `CtxV2_ImgOnly` vs `CtxV1`：纯图像上下文和纯文本上下文哪个更强

**注意：** 本实验使用 `complete_multimodal_context.json`（contextV2 输出），
需先运行 `build_multimodal_context_v2.ipynb` 生成该文件。

## §1  Environment Setup

In [ ]:
import sys, os, json, pathlib, importlib

# ── Mount Google Drive (Colab) ───────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    COLAB = True
except ImportError:
    COLAB = False
    print("[Local mode] Drive not mounted.")

# ── Locate code directory ────────────────────────────────────────────────────
if COLAB:
    CODE_DIR = pathlib.Path("/content/drive/MyDrive/EyeSeq/code/GazeDecoderV3")
else:
    CODE_DIR = pathlib.Path(".").resolve()

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# ── Force-reload shared/* in dependency order ────────────────────────────────
_RELOAD_ORDER = [
    "shared.config",
    "shared.features",
    "shared.dataset",
    "shared.models",
    "shared.training",
    "shared.viz",
]
for _mod in _RELOAD_ORDER:
    if _mod in sys.modules:
        importlib.reload(sys.modules[_mod])

import shared.config as _cfg
print(f"CODE_DIR = {CODE_DIR}")
print(f"COLAB    = {COLAB}")
print(f"EPOCHS   = {_cfg.EPOCHS}  |  FEAT_DIM = {_cfg.FEAT_DIM}")


In [ ]:
if COLAB:
    %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
    %pip install -q scikit-learn xgboost lightgbm sentence-transformers tqdm seaborn scipy
    print("✅ Packages installed.")


In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from shared.config import (
    SEED, GAZE_DIR, ECONTEXT_PATH,
    ABL_DIR, DS_CACHE_DIR,
    WINDOW_SIZE, STRIDE, HIT_THRESHOLD, FEAT_DIM,
    set_global_seed,
)
from shared.dataset  import build_dataset, build_dataset_v2, get_loso_splits
from shared.models   import Bchan_Spatial_CtxSA, ModelSpec
from shared.training import run_loso, run_all_models
from shared.viz      import (results_to_df, plot_f1_leaderboard, plot_prf_grouped,
                              significance_table, scott_knott_esd, plot_scott_knott)

set_global_seed()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"SEED={SEED} | torch={torch.__version__} | device={DEVICE}")

# ── CtxV2 专用路径 ────────────────────────────────────────────────────────────
EYESEQ_ROOT  = Path("/content/drive/MyDrive/EyeSeq") if COLAB else Path(GAZE_DIR).parent.parent
V2_CTX_PATH  = EYESEQ_ROOT / "contextV2" / "complete_multimodal_context.json"
CTX_V2_ARCHIVE = ABL_DIR.parent / "ablation_ctx_v2"
CTX_V2_ARCHIVE.mkdir(parents=True, exist_ok=True)

print(f"V2 context json : {V2_CTX_PATH}")
print(f"Archive dir     : {CTX_V2_ARCHIVE}")
print(f"V2 json exists  : {V2_CTX_PATH.exists()}")


## §2  构建 4 个对比数据集

每个数据集只有 `ctx_mode` 不同，其余（LOSO 切分、gaze 特征、cache 机制）完全相同。
首次运行会重新 build 并缓存，后续直接从 pkl 加载。

> **Key 格式说明（重要）：**  
> `complete_multimodal_context.json` 的 key 格式与旧版 `complete_econtext.json` **完全相同**：  
> `"componentInfo|src_rel_path:line_number"`，例如 `"aoi logo|views/services/AncientRead.vue:3"`  
> `AOI.csv` 的 `src_index` 列也带行号，因此 `dataset.py` 中直接用 `hit['src']` 拼接即可，**无需 strip 行号**。

> **`v1_compat` 模式说明：**  
> 读取新 JSON 中的 `embed_func`（384d）+ `embed_code`（768d）→ 拼接 1152d → 随机正交投影到 768d。  
> 与旧版 `complete_econtext.json` 的字段含义对应（embed_func ≈ embed_text，embed_code 同名但从 384d 升为 768d CodeBERT）。  
> 真正的历史基线（旧 JSON 直接填充）仍由 `ablation.ipynb` 的 `Bchan_Spatial_CtxSA` 代表（F1=0.9467）。


In [ ]:
# ── §2 前置检查：缓存状态一览 ────────────────────────────────────────────────
# 三种缓存文件名格式（互不冲突，无需担心误覆盖）：
#
#  ① EyeSeqDataset (旧版)  : eyeseq_w64_s32_thr10_d786.pkl
#  ② EyeSeqDatasetV2       : eyeseq_w64_s32_thr10_d786_v2_{ctx_mode}.pkl
#  ③ EyeSeqDatasetV2_Wide  : eyeseq_w64_s32_thr10_d2322_wide.pkl  ← 全新，不冲突
#
# 需要删除的情况：
#   ② 类缓存若是在 "key strip行号 bug" 修复前生成的 → 内容是零向量 → 必须删除
#   ③ 类缓存从未生成过 → 无需任何操作
#
# 判断方法：下方代码会打印各缓存的"ctx 非零率"，若 <50% 即为污染缓存。
# 若需强制删除，取消 FORCE_REBUILD=True 注释并重跑本 Cell。

import glob as _glob
import os as _os
import json as _j
import pickle as _pk
import numpy as _np
import torch as _t

FORCE_REBUILD = False   # ← 改为 True 可强制删除所有 V2 缓存并重建

CTX_MODES_LIST = ["v1_compat", "full", "no_img", "img_only"]
_sem = slice(2, 770)   # _S_SEMANTIC — 与 models.py 一致

print("=" * 60)
print("  数据集缓存状态检查")
print("=" * 60)

stale = []
for _mode in CTX_MODES_LIST:
    _name = f"eyeseq_w{WINDOW_SIZE}_s{STRIDE}_thr{HIT_THRESHOLD}_d{FEAT_DIM}_v2_{_mode}.pkl"
    _path = DS_CACHE_DIR / _name
    if not _path.exists():
        print(f"  [CtxV2/{_mode}] ⬜ 无缓存，将重新 build")
        continue
    # 采样检查非零率（只读前 200 个样本，避免耗时）
    try:
        with open(_path, "rb") as _f:
            _samples = _pk.load(_f)
        _check = _samples[:200]
        _nz = sum(1 for s in _check
                  if _t.from_numpy(s["x"])[:, _sem].abs().sum().item() > 0)
        _rate = _nz / len(_check) * 100
        if _rate < 50:
            _tag = f"❌ 污染缓存（非零率={_rate:.0f}%），需删除"
            stale.append(str(_path))
        else:
            _tag = f"✅ 正常（非零率={_rate:.0f}%）"
        print(f"  [CtxV2/{_mode}] {_tag}  ({len(_samples):,} windows)")
    except Exception as _e:
        print(f"  [CtxV2/{_mode}] ⚠️  读取失败: {_e}")
        stale.append(str(_path))

# Wide 缓存（d2322）
from shared.dataset import WIDE_FEAT_DIM as _WFD
_wide_name = f"eyeseq_w{WINDOW_SIZE}_s{STRIDE}_thr{HIT_THRESHOLD}_d{_WFD}_wide.pkl"
_wide_path = DS_CACHE_DIR / _wide_name
if _wide_path.exists():
    print(f"  [Wide]         ✅ 已有缓存: {_wide_name}")
else:
    print(f"  [Wide]         ⬜ 无缓存，将重新 build（首次约 5-10 分钟）")

# 旧版 V1 缓存
_v1_name = f"eyeseq_w{WINDOW_SIZE}_s{STRIDE}_thr{HIT_THRESHOLD}_d{FEAT_DIM}.pkl"
_v1_path = DS_CACHE_DIR / _v1_name
print(f"  [V1 旧版]      {'✅ 已有' if _v1_path.exists() else '⬜ 无缓存'}: {_v1_name}")

print()

# LOSO 结果目录
print("  LOSO 已有结果：")
_any_result = False
for _exp in ["CtxV1", "CtxV2_Full", "CtxV2_NoImg", "CtxV2_ImgOnly"]:
    _rpt = CTX_V2_ARCHIVE / _exp / "final_report.json"
    if _rpt.exists():
        _s = _j.loads(_rpt.read_text()).get("summary", {})
        print(f"    📋 {_exp}: F1_issue={_s.get('f1_issue', float('nan')):.4f}")
        _any_result = True
if not _any_result:
    print("    （无已有结果）")

# 自动或手动清理污染缓存
if stale:
    print(f"\n⚠️  发现 {len(stale)} 个污染缓存:")
    for _f in stale:
        print(f"   {_f}")
    if FORCE_REBUILD:
        for _f in stale:
            _os.remove(_f)
            print(f"   🗑️  已删除: {_os.path.basename(_f)}")
    else:
        print("\n   → 将 FORCE_REBUILD=True 并重跑本 Cell 可自动删除；")
        print("     或手动删除后继续。")
else:
    print("\n✅ 无污染缓存，可直接运行 §3。")


In [ ]:
# ── 4 个数据集（共用 cache_dir，文件名含 ctx_mode 后缀自动区分） ──────────────

CTX_MODES = {
    "CtxV1"        : "v1_compat",   # G1 func + G2 code (对齐旧版字段结构)
    "CtxV2_Full"   : "full",        # G1+G2+G3a+G3b+G4+G5 全模态
    "CtxV2_NoImg"  : "no_img",      # G1+G2+G3a+G3b  纯文字
    "CtxV2_ImgOnly": "img_only",    # G4 patch + G5 page 纯图像
}

datasets = {}
for exp_name, mode in CTX_MODES.items():
    print(f"\n{'='*55}")
    print(f"Building dataset: {exp_name}  (ctx_mode='{mode}')")
    print(f"{'='*55}")
    ds = build_dataset_v2(
        gaze_dir         = str(GAZE_DIR),
        v2_econtext_path = str(V2_CTX_PATH),
        cache_dir        = str(DS_CACHE_DIR),
        ctx_mode         = mode,
        window_size      = WINDOW_SIZE,
        stride           = STRIDE,
        hit_threshold    = HIT_THRESHOLD,
    )
    # run_loso 内部会自己调 get_loso_splits，这里只做验证用
    splits = get_loso_splits(ds)
    datasets[exp_name] = ds   # 只存 ds，splits 由 run_loso 内部处理
    print(f"  → {len(ds):,} windows, {len(splits)} LOSO folds")
    print(f"     feature shape: {ds[0][0].shape}  (expected [T={WINDOW_SIZE}, D={FEAT_DIM}])")

    # ── 关键健康检查：确认上下文向量非零比率 ──────────────────────────────────
    import torch as _t
    _sem_slice = slice(2, 770)   # _S_SEMANTIC，与 models.py 完全一致
    _n_nonzero = sum(
        1 for s in ds.samples
        if _t.from_numpy(s["x"])[:, _sem_slice].abs().sum().item() > 0
    )
    _hit_rate = _n_nonzero / len(ds) * 100
    _status = "✅" if _hit_rate > 50 else "❌ 严重异常！上下文几乎全是零向量"
    print(f"     ctx non-zero windows: {_n_nonzero}/{len(ds)} ({_hit_rate:.1f}%)  {_status}")


## §3  运行 LOSO 对比实验

对每个 `ctx_mode` 数据集分别训练 `Bchan_Spatial_CtxSA`。  
结果缓存到 `archive/ablation_ctx_v2/<exp_name>/fold_NN.json`，**可随时续跑**。

In [ ]:
import json as _json
from shared.training import run_loso
from shared.models import ModelSpec, Bchan_Spatial_CtxSA

# 固定使用最优架构变体，唯一变量是 ctx_mode（体现在 dataset 里）
CTX_V2_SPEC = ModelSpec(
    name  = "Bchan_Spatial_CtxSA",
    kind  = "dl",
    build = lambda: Bchan_Spatial_CtxSA(),
)

ctx_v2_results = {}   # exp_name → result dict

for exp_name, ds in datasets.items():
    print(f"\n{'='*55}")
    print(f"▶  {exp_name}  (ctx_mode='{CTX_MODES[exp_name]}')")
    print(f"{'='*55}")

    exp_archive = CTX_V2_ARCHIVE / exp_name
    exp_archive.mkdir(parents=True, exist_ok=True)

    result = run_loso(
        spec      = CTX_V2_SPEC,
        ds        = ds,
        cache_dir = exp_archive,
        verbose   = True,
    )
    ctx_v2_results[exp_name] = result

    summary = result.get("summary", result)
    f1  = summary.get("f1_issue", float("nan"))
    f1m = summary.get("f1_macro", float("nan"))
    print(f"  → F1_issue={f1:.4f}   F1_macro={f1m:.4f}")

print("\n✅ All experiments complete.")


## §4  结果对比与决策

In [ ]:
# ── 辅助：从 final_report 提取各 fold 的 f1_issue ────────────────────────────
def get_fold_f1s(results: dict, exp_name: str) -> list:
    """run_loso 返回 final_report，fold_metrics 是 per-fold metric dict 列表。"""
    res = results.get(exp_name, {})
    return [m.get("f1_issue", float("nan")) for m in res.get("fold_metrics", [])]

def get_summary_val(results: dict, exp_name: str, key: str) -> float:
    res = results.get(exp_name, {})
    return res.get("summary", {}).get(key, float("nan"))

# ── 数值汇总表 ────────────────────────────────────────────────────────────────
rows = []
for exp_name in CTX_MODES:
    if exp_name not in ctx_v2_results:
        continue
    rows.append({
        "experiment":  exp_name,
        "ctx_mode":    CTX_MODES[exp_name],
        "f1_issue":    round(get_summary_val(ctx_v2_results, exp_name, "f1_issue"), 4),
        "f1_macro":    round(get_summary_val(ctx_v2_results, exp_name, "f1_macro"), 4),
        "precision":   round(get_summary_val(ctx_v2_results, exp_name, "p_issue"), 4),
        "recall":      round(get_summary_val(ctx_v2_results, exp_name, "r_issue"), 4),
    })

summary_df = pd.DataFrame(rows).sort_values("f1_issue", ascending=False)
print("=" * 65)
print("  Multimodal Context Ablation — LOSO Summary")
print("=" * 65)
display(summary_df.reset_index(drop=True))

# ── 与历史基线（旧版 CtxSA）对比 ─────────────────────────────────────────────
HIST_BASELINE_F1 = 0.9467   # Bchan_Spatial_CtxSA 在旧版完整消融中的最优 F1_issue
print(f"\n参考值: Bchan_Spatial_CtxSA (旧 JSON) = {HIST_BASELINE_F1:.4f}")
print()
for _, row in summary_df.iterrows():
    delta = row["f1_issue"] - HIST_BASELINE_F1
    sign  = "▲" if delta > 0.001 else ("▼" if delta < -0.001 else "≈")
    print(f"  {sign} {row['experiment']:<18}  F1_issue={row['f1_issue']:.4f}  ({delta:+.4f})")


In [ ]:
# ── 可视化：F1_issue 对比条形图 ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

colors = ["#5B9BD5", "#ED7D31", "#70AD47", "#FFC000"]
bars = ax.barh(
    summary_df["experiment"],
    summary_df["f1_issue"],
    color=colors[: len(summary_df)],
    height=0.5,
)
ax.axvline(HIST_BASELINE_F1, color="red", linestyle="--", linewidth=1.5,
           label=f"V1 baseline ({HIST_BASELINE_F1:.4f})")
ax.set_xlabel("F1_issue (LOSO mean)")
ax.set_title("Multimodal Context Ablation — CtxV2 vs CtxV1")
ax.legend()
ax.set_xlim(max(0, summary_df["f1_issue"].min() - 0.03), 1.0)
for bar, val in zip(bars, summary_df["f1_issue"]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=9)
plt.tight_layout()
fig.savefig(str(CTX_V2_ARCHIVE / "fig_ctx_v2_comparison.png"), dpi=150)
plt.show()


In [ ]:
# ── 显著性检验（Wilcoxon 配对，per-fold F1_issue）────────────────────────────
# 比较 CtxV2_Full vs CtxV1  （最关键的一对）
from scipy import stats

baseline_f1s = get_fold_f1s(ctx_v2_results, "CtxV1")
full_f1s     = get_fold_f1s(ctx_v2_results, "CtxV2_Full")

if len(baseline_f1s) >= 2 and len(full_f1s) >= 2:
    stat, p_val = stats.wilcoxon(full_f1s, baseline_f1s, alternative="two-sided")
    print(f"Wilcoxon test (CtxV2_Full vs CtxV1):")
    print(f"  per-fold F1_issue: CtxV1    = {[round(v,4) for v in baseline_f1s]}")
    print(f"                     CtxV2_Full = {[round(v,4) for v in full_f1s]}")
    print(f"  statistic={stat:.3f}  p={p_val:.4f}")
    if p_val < 0.05:
        winner = "CtxV2_Full" if np.mean(full_f1s) > np.mean(baseline_f1s) else "CtxV1"
        print(f"  ✅ Significant (p<0.05) — {winner} is better")
    else:
        print("  ⚠️  Not significant (p≥0.05) — difference may be noise")
else:
    print("Not enough fold data for statistical test (need ≥2 folds).")

# ── 决策建议 ──────────────────────────────────────────────────────────────────
best_exp = summary_df.iloc[0]["experiment"]
best_f1  = summary_df.iloc[0]["f1_issue"]
delta    = best_f1 - HIST_BASELINE_F1

print(f"\n{'='*55}")
print(f"决策建议")
print(f"{'='*55}")
if delta > 0.005:
    print(f"  ✅ 推荐继续多模态路线")
    print(f"     最优实验：{best_exp}  (+{delta:.4f} vs 旧基线)")
    print(f"     建议：扩展 config.py/dataset.py 支持原生宽维度，")
    print(f"           设计专用 MultiModalCtxProj 替代随机投影")
elif delta > 0.001:
    print(f"  ⚠️  提升微弱 (+{delta:.4f})，建议检查投影质量后再决定")
    print(f"     可尝试：减少 proj_seed 随机性 / 换用 PCA 初始化投影")
else:
    print(f"  ❌ 多模态上下文无明显提升（delta={delta:+.4f}）")
    print(f"     建议：放弃 CtxV2 路线，沿用旧版 complete_econtext.json")


In [ ]:
# ── 保存汇总 JSON 到 archive ──────────────────────────────────────────────────
summary_out = {
    "hist_baseline_f1": HIST_BASELINE_F1,
    "experiments": {
        exp: {
            "ctx_mode":       CTX_MODES[exp],
            "f1_issue":       get_summary_val(ctx_v2_results, exp, "f1_issue"),
            "f1_macro":       get_summary_val(ctx_v2_results, exp, "f1_macro"),
            "fold_f1_issues": get_fold_f1s(ctx_v2_results, exp),
        }
        for exp in CTX_MODES
        if exp in ctx_v2_results
    }
}
out_path = CTX_V2_ARCHIVE / "ctx_v2_summary.json"
out_path.write_text(_json.dumps(summary_out, indent=2))
print(f"✅ Summary saved → {out_path}")
print(_json.dumps(summary_out, indent=2))


## §5  诊断与后续路线

### 实验结果汇总（已验证）

| 实验 | 数据来源 | ctx 维度 | F1_issue | Δ vs 旧基线 |
|---|---|---|---|---|
| Baseline（旧JSON） | `complete_econtext.json` | 768d（直接） | **0.9467** | +0.0000 |
| CtxV2_Full | 新JSON，全模态投影 | 2688d→768d | 0.9413 | -0.0054 |
| CtxV2_NoImg | 新JSON，纯文字投影 | 1920d→768d | 0.9409 | -0.0058 |
| CtxV2_ImgOnly | 新JSON，纯图像投影 | 768d→768d | 0.9407 | -0.0060 |
| CtxWide（无投影） | 旧+新JSON，拼接 | 2304d（直接） | 0.9372 | -0.0095 |
| CtxV1（v1_compat） | 新JSON，旧字段投影 | 1152d→768d | 0.9248 | -0.0219 |

---

### 为什么旧基线仍然最优？

**旧基线是唯一零信息损失的方案**：`embed_text(384d) + embed_code(384d)` = 768d，直接填入 `[2:770]`，无任何变换。所有新方案都引入了额外瓶颈：

| 方案 | 额外损耗来源 |
|---|---|
| CtxV1 | 1152d→768d 随机正交投影，损失 ~33% |
| CtxV2_Full/NoImg/ImgOnly | 更大维度→768d 投影，损失更多 |
| CtxWide | `ctx_proj` 从 768→2304d，参数量增加 ×3，训练数据不足以支撑收敛 |

---

### 新模态无效的证据

三组投影版（Full / NoImg / ImgOnly）的 F1 差距 < 0.001：

- OCR 文本（G3a/G3b）单独贡献：≈ 0
- 图像嵌入（G4/G5）单独贡献：≈ 0
- **新模态没有带来可观测的信息增益**，无论有无图像结果几乎相同。

Wide 方案（无投影）反而比 Full 投影版更差（-0.0095 vs -0.0054），说明问题不是投影损失，而是新模态本身信号质量有限 + 参数量增大导致训练效率下降。

---

### 参考：旧消融实验——各模块贡献（`ablation.ipynb` 已验证，n_folds=6）

以下为旧消融实验（baseline=Bchan_Spatial_CtxSA，F1=0.9467）中各消融组的统计结果：

| 消融组 | 含义 | mean F1 | Δ（↓） | p 值 | Cohen's d |
|---|---|---|---|---|---|
| `CtxSA_NoCtx` | 去掉全部 ctx 流 | 0.8148 | **-0.1319** | 0.023 | 1.33 |
| `CtxSA_NoText` | 去掉 embed_text（保留 code） | 0.9197 | -0.0270 | 0.0018 | **2.45** |
| `CtxSA_NoBehav` | 去掉 gaze 行为流 | 0.9345 | -0.0122 | 0.0032 | 2.17 |
| `CtxSA_NoIIB` | 去掉 IIB（输入侧 ctx 注入） | 0.9322 | -0.0145 | 0.023 | 1.33 |
| `CtxSA_NoOIB` | 去掉 OIB（输出侧 ctx 精炼） | 0.9414 | -0.0053 | 0.065 | 0.96 |
| `CtxSA_NoCode` | 去掉 embed_code（保留 text） | 0.9398 | -0.0069 | 0.072 | 0.93 |
| `CtxSA_1StageProj` | 两阶段 ctx_proj→单阶段 | 0.9395 | -0.0072 | 0.250 | — |

**关键结论：**
- **ctx 流整体不可缺（-0.13）**，是模型最重要的外部信号来源
- **embed_text 贡献显著大于 embed_code**（-0.027 vs -0.007，p=0.0018，d=2.45）
- **IIB 贡献（输入侧注入）显著（-0.015，p=0.023）**；OIB（输出侧）贡献临界（p=0.065）
- **gaze 行为流本身也关键**（去掉后 -0.012，p=0.003）
- ctx_proj 两阶段 vs 单阶段无显著差异（p=0.25）

> 这意味着新模态要想超越旧基线，**必须能提供比现有 embed_text 更强的语义信号**。而 OCR / 截图嵌入显然未能做到这一点（Full 组 Δ=-0.005）。

---

### 结论与决策

❌ **放弃多模态扩展路线，沿用旧版 `complete_econtext.json`。**

旧基线（F1=0.9467）已是本框架在当前数据规模下的性能上限。新模态（OCR、截图嵌入）未能带来净收益，继续投入的边际回报极低。

后续方向应聚焦于：
1. **数据层面**：扩充标注数据（更多受试者 / 更多项目）
2. **序列建模层面**：改进 backbone 或位置编码
3. **标签质量层面**：重新审视 `issue` 标签的一致性


## §6  Wide 实验：Bchan_Spatial_CtxSA_Wide（原生宽维度，无投影）

In [ ]:
# ── §6-A: 构建 Wide 数据集 ────────────────────────────────────────────────────
# EyeSeqDatasetV2_Wide 读取两个 JSON：
#   旧 JSON (complete_econtext.json)    → embed_text(384) + embed_code(384) = 768d
#   新 JSON (complete_multimodal_context.json) → G3a+G3b+G4+G5 = 4×384 = 1536d
# 拼接为 2304d ctx 块，直接填入 [2:2306]，无任何投影。
#
# 模型 Bchan_Spatial_CtxSA_Wide:
#   ctx_proj 接受 2304d 输入（vs 原版 768d），其余架构完全一致。

from shared.dataset import (
    EyeSeqDatasetV2_Wide, build_dataset_v2_wide,
    WIDE_FEAT_DIM, WIDE_CTX_DIM,
    get_loso_splits,
)
from shared.models import Bchan_Spatial_CtxSA_Wide, ModelSpec

print(f"Wide 数据集特征维度: {WIDE_FEAT_DIM}d  (ctx={WIDE_CTX_DIM}d)")

# ── 路径确认 ──────────────────────────────────────────────────────────────────
print(f"旧 JSON:  {ECONTEXT_PATH}")
print(f"新 JSON:  {V2_CTX_PATH}")
print(f"Archive:  {CTX_V2_ARCHIVE}")

# ── 检查 Wide cache ────────────────────────────────────────────────────────────
import glob as _glob
wide_caches = _glob.glob(str(DS_CACHE_DIR / f"*_d{WIDE_FEAT_DIM}_wide*.pkl"))
if wide_caches:
    print(f"\n📦 已有 Wide cache:")
    for c in wide_caches:
        print(f"   {c}")
else:
    print("\n⚠️  无 Wide cache，首次运行将从头构建（需要 ~5-10 分钟）")


In [ ]:
# ── §6-B: 构建 Wide 数据集 + 健康检查 ─────────────────────────────────────────
import torch as _t

ds_wide = build_dataset_v2_wide(
    gaze_dir         = str(GAZE_DIR),
    econtext_path    = str(ECONTEXT_PATH),
    v2_econtext_path = str(V2_CTX_PATH),
    cache_dir        = str(DS_CACHE_DIR),
    window_size      = WINDOW_SIZE,
    stride           = STRIDE,
    hit_threshold    = HIT_THRESHOLD,
)

splits_wide = get_loso_splits(ds_wide)
print(f"\nWide 数据集：{len(ds_wide):,} windows，{len(splits_wide)} LOSO folds")
print(f"feature shape: {ds_wide[0][0].shape}  (expected [T={WINDOW_SIZE}, D={WIDE_FEAT_DIM}])")

# ── 健康检查 1：旧版 768d 槽位非零率（[2:770]）──────────────────────────────
_old_ctx = slice(2, 770)
_n_old   = sum(1 for s in ds_wide.samples
               if _t.from_numpy(s["x"])[:, _old_ctx].abs().sum().item() > 0)
_r_old   = _n_old / len(ds_wide) * 100
_s_old   = "✅" if _r_old > 50 else "❌ 旧 ctx 几乎全零！"
print(f"\n旧 ctx [2:770] 非零窗口: {_n_old}/{len(ds_wide)} ({_r_old:.1f}%)  {_s_old}")

# ── 健康检查 2：新通道非零率（[770:2306]）──────────────────────────────────
_new_ctx = slice(770, 2306)
_n_new   = sum(1 for s in ds_wide.samples
               if _t.from_numpy(s["x"])[:, _new_ctx].abs().sum().item() > 0)
_r_new   = _n_new / len(ds_wide) * 100
_s_new   = "✅" if _r_new > 50 else "❌ 新 ctx 几乎全零！"
print(f"新 ctx [770:2306] 非零窗口: {_n_new}/{len(ds_wide)} ({_r_new:.1f}%)  {_s_new}")


In [ ]:
# ── §6-C: 运行 Wide LOSO 实验 ─────────────────────────────────────────────────
# Bchan_Spatial_CtxSA_Wide 与旧版架构完全相同，仅 ctx_proj 接受 2304d 输入。
# 旧版 Bchan_Spatial_CtxSA 不受任何影响。

WIDE_ARCHIVE = CTX_V2_ARCHIVE.parent / "ctx_v2_wide"   # 与 CTX_V2_ARCHIVE 同级
WIDE_ARCHIVE.mkdir(parents=True, exist_ok=True)

WIDE_SPEC = ModelSpec(
    name  = "Bchan_Spatial_CtxSA_Wide",
    kind  = "dl",
    build = lambda: Bchan_Spatial_CtxSA_Wide(),
)

# 检查是否有已有结果
wide_report = WIDE_ARCHIVE / "final_report.json"
if wide_report.exists():
    print(f"⚡ 已有 Wide 实验结果：{wide_report}")
    with open(wide_report) as _f:
        wide_result = _json.load(_f)
    print(f"   F1_issue (mean) = {wide_result.get('summary', {}).get('f1_issue', 'N/A'):.4f}")
else:
    print("▶️  开始 Wide LOSO 实验…")
    wide_result = run_loso(
        spec     = WIDE_SPEC,
        ds       = ds_wide,
        cache_dir= WIDE_ARCHIVE,
        verbose  = True,
    )

    # 保存结果
    with open(wide_report, "w") as _f:
        _json.dump(wide_result, _f, indent=2)
    print(f"\n✅ Wide 结果已保存 → {wide_report}")
    print(f"   F1_issue (mean) = {wide_result.get('summary', {}).get('f1_issue', 'N/A'):.4f}")


In [ ]:
# ── §6-D: 对比汇总（含 Wide）────────────────────────────────────────────────

HIST_BASELINE_F1 = 0.9467   # Bchan_Spatial_CtxSA + 旧 JSON

def _load_summary_f1(report_path):
    """从 final_report.json 读取 f1_issue 均值."""
    p = Path(report_path)
    if not p.exists():
        return float("nan")
    with open(p) as f:
        r = _json.load(f)
    return r.get("summary", {}).get("f1_issue", float("nan"))

def _load_fold_f1s_from_report(report_path):
    p = Path(report_path)
    if not p.exists():
        return []
    with open(p) as f:
        r = _json.load(f)
    return [m.get("f1_issue", float("nan")) for m in r.get("fold_metrics", [])]

# 汇总所有实验
compare_rows = [
    {"实验": "Baseline (旧JSON)", "ctx 维度": "768d (直接)", "F1_issue": HIST_BASELINE_F1, "备注": "ablation.ipynb 最优"},
]
for exp_name, mode in CTX_MODES.items():
    f1 = get_summary_val(ctx_v2_results, exp_name, "f1_issue") if exp_name in ctx_v2_results else float("nan")
    compare_rows.append({
        "实验": exp_name,
        "ctx 维度": f"→768d (proj, {mode})",
        "F1_issue": round(f1, 4),
        "备注": "EyeSeqDatasetV2 (随机投影)"
    })

# Wide 结果（直接从 wide_result 变量读，或从 report 文件读）
try:
    wide_f1 = wide_result.get("summary", {}).get("f1_issue", float("nan"))
except NameError:
    wide_f1 = _load_summary_f1(WIDE_ARCHIVE / "final_report.json")

compare_rows.append({
    "实验": "CtxWide (无投影)",
    "ctx 维度": "2304d (直接)",
    "F1_issue": round(wide_f1, 4),
    "备注": "EyeSeqDatasetV2_Wide"
})

import pandas as _pd
compare_df = _pd.DataFrame(compare_rows)
compare_df["Δ vs 旧基线"] = compare_df["F1_issue"].apply(
    lambda v: f"{v - HIST_BASELINE_F1:+.4f}" if not (_pd.isna(v) if not isinstance(v, float) else v != v) else "N/A"
)
print("=" * 75)
print("  完整对比：旧基线 / CtxV1-V2（投影） / CtxWide（无投影）")
print("=" * 75)
display(compare_df)
